# 1D Heat Equation: Nonhomogeneous Dirichlet Conditions

In this notebook we illustrate the performance of the explicit finite differences method for solving the 1D heat equation
\begin{align*}
\frac{\partial u}{\partial t} &= \frac{\partial^2 u}{\partial x^2}\\
u(0,t)&=5t\\
u(1,t)&=-5t\\
u(x,0)&=f(x)
\end{align*}
for $x\in(0,1)$ and $t\ge0$. We set the diffusion coefficient $D=1$ and the initial condition $f(x)=1-4(x-1/2)^2$, and compare our numerical approximations with the exact solution.

We also set the parameters for the numerical schemes as follows:

- Spatial step-size: $\Delta x = 1/10$.
- Temporal step-size: $\Delta t = 0.001$.
- Stability parameter: $s=\frac{D\Delta t}{(\Delta x)^2}=0.1$.

We will compare the approximate solution with the exact solution at $t=0.01$. We also show an animation of the change in temperature through the given time interval using the exact solution and the approximations.

## Libraries

In [ ]:
import numpy as np
from numpy.linalg import solve, norm
from numpy import pi,exp,sin,cos
import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib.animation as animation
import pandas as pd

## $t\in [0, 0.1], \Delta t = .001, \Delta x = 0.1, s = 0.1$

### Exact solution

In [ ]:
udnh_f = np.loadtxt("uexmpdnh.csv", delimiter=",")
udnh_f = udnh_f.T

### Discretized exact solution

In [ ]:
udnh = np.loadtxt("uexmpdnh1.csv", delimiter=",")
udnh = udnh.T

### Parameter setup

In [ ]:
xmin, xmax = 0, 1
N = 10
dx = (xmax-xmin)/N
x = np.linspace(xmin,xmax,N+1)

tmin, tmax = 0, 0.1
dt = 0.001
M = int((tmax-tmin)/dt)
t = np.linspace(tmin,tmax,M+1)

D = 1
s = D*dt/(dx*dx)

X,T = np.meshgrid(x,t)
vmin, vmax = 0., 1.
levels = np.linspace(vmin, vmax, 21)

In [ ]:
(xmin,xmax),(tmin,tmax),N,M,D,dx,dt,s

### Initial and boundary conditions

In [ ]:
def f(x):
    return 1-4*(x-1/2)**2

def g0(t):
    return 5*t

def g1(t):
    return -5*t

### Initial graphs

In [ ]:
xx = np.linspace(xmin,xmax,num=101)
tt = np.linspace(tmin,tmax,num=1001)
XX,TT = np.meshgrid(xx,tt)

In [ ]:
fig,ax = plt.subplots(subplot_kw={"projection": "3d"},figsize=(10,10), layout='tight')
ax.plot_surface(XX, TT, udnh_f.T,cmap=cm.viridis)
ax.set(title='Exact solution',xlabel='x',ylabel='t',zlabel='u')
ax.set_box_aspect(None, zoom=0.87)
plt.show()

In [ ]:
fig,ax = plt.subplots()
ct = ax.contourf(XX, TT, udnh_f.T, levels = levels, extend = 'both');
fig.colorbar(ct)
ax.set(title='Temperature contour map',xlabel='x',ylabel='t')
plt.show()

In [ ]:
fig,ax = plt.subplots()
ax.plot(xx,f(xx));
ax.set(title="Initial condition",xlabel='x',ylabel='u(x,0)')
ax.grid()
plt.show()

In [ ]:
fig,ax = plt.subplots()
ax.plot(x,f(x));
ax.set(title="Discretized initial condition",xlabel='x',ylabel='u(x,0)')
ax.grid()
plt.show()

In [ ]:
fig,ax = plt.subplots()
for k in range(0,6):
    ax.plot(xx,udnh_f[:,200*k],label=f't = {t[20*k]}')
ax.set(title='Temperature variation over time',xlabel='x',ylabel='u')
ax.grid()
ax.legend()
plt.show()

### Animation of exact solution

In [ ]:
fig, ax = plt.subplots()
heat_exact, = ax.plot(xx,udnh_f[:,0]);

xlim = (-0.1, 1.1)
ylim = (-0.6, 1.1)
ax.set(xlabel='x',ylabel='u(x,t)',xlim=xlim,ylim=ylim)
ax.grid()

def animate(k):
    heat_exact.set_ydata(udnh_f[:,10*k])
    ax.set(title=f"Exact solution at t = {k*dt/10:.4f}")
    return heat_exact

ani = animation.FuncAnimation(fig, animate, frames=101)#, interval=500)

plt.show()

ani.save('heat_udnh_soln.gif',fps=24)
print("Animation saved successfully as heat_udnh_soln.gif")

### Forward Euler

#### Computations

In [ ]:
u = np.zeros((N+1,M+1))
u[:,0] = f(x)
u[0,:] = g0(t)
u[N,:] = g1(t)

In [ ]:
for k in range(0,M):
    u[1:-1,k+1] = u[1:-1,k]+s*(u[2:,k]-2*u[1:-1,k]+u[:-2,k])

In [ ]:
u_euler_fwd = u[:,:]

In [ ]:
error_e = u_euler_fwd[:,:]-udnh[:,:]

In [ ]:
print(f'Linf error = {norm(error_e[:,M],ord=np.inf)}')

In [ ]:
df_euler_fwd = pd.DataFrame(u_euler_fwd.T, columns = x, index = t)

In [ ]:
df_euler_fwd

In [ ]:
df_euler_fwd_error = pd.DataFrame(error_e.T, columns = x, index = t)

In [ ]:
df_euler_fwd_error

#### Graphs

In [ ]:
fig,ax = plt.subplots(subplot_kw={"projection": "3d"},figsize=(10,10), layout='tight')
ax.plot_surface(X, T, u_euler_fwd.T,cmap=cm.viridis)
ax.set(title='Forward Euler approximation',xlabel='x',ylabel='t',zlabel='u')
ax.set_box_aspect(None, zoom=0.87)
plt.show()

In [ ]:
fig,ax = plt.subplots()
ct = ax.contourf(X, T, u_euler_fwd.T, levels = levels, extend = 'both');
fig.colorbar(ct)
ax.set(title='Temperature contour map',xlabel='x',ylabel='t')
plt.show()

In [ ]:
fig,ax = plt.subplots()
for k in range(0,M+1,M//5):
    ax.plot(x,u_euler_fwd[:,k],label=f't = {t[k]}')
ax.set(title='Temperature variation over time',xlabel='x',ylabel='u')
ax.grid()
ax.legend()
plt.show()

In [ ]:
fig,ax = plt.subplots()
ax.plot(x,u_euler_fwd[:,M],label=f'Approximation at t = {t[M]}')
ax.plot(x,udnh[:,M],label=f'Exact solution at t = {tmax}')
ax.set(title='Comparison between exact and approximate solution',xlabel='x',ylabel='u')
ax.grid()
ax.legend()
plt.show()

In [ ]:
fig,ax = plt.subplots()
ax.plot(x,abs(error_e[:,M]))
ax.set(title='Absolute error',xlabel='x',ylabel='error')
ax.grid()
plt.show()

In [ ]:
fig, ax = plt.subplots()

heat_approx, = ax.plot(x,u_euler_fwd[:,0],label='Forward Euler approximation');
heat_exact, = ax.plot(x,udnh[:,0],linestyle='-.',label='Exact solution');

ax.legend()
ax.set(xlabel='x',ylabel='u(x,t)',xlim=xlim,ylim=ylim)
ax.grid()

def animate(k):
    heat_approx.set_ydata(u_euler_fwd[:,k])
    heat_exact.set_ydata(udnh[:,k])
    ax.set(title=f"Exact solution vs Forward Euler approximation at t = {k*dt:.3f}")
    return heat_approx,heat_exact

ani = animation.FuncAnimation(fig, animate, frames=M+1)#, interval=500)

plt.show()

ani.save('heat_udnh_fwd.gif', fps=24)
print("Animation saved successfully as heat_udnh_fwd.gif")

In [ ]:
fig, ax = plt.subplots()

error, = ax.plot(x,abs(error_e[:,0]));

#ax.legend()
ax.set(xlabel='x',ylabel='error',xlim=xlim,ylim=[0,1.5e-3])
ax.grid()

def animate(k):
    error.set_ydata(abs(error_e[:,k]))
    ax.set(title=f"Error propagation. t = {k*dt:.3f}")
    return error

ani = animation.FuncAnimation(fig, animate, frames=M+1)#, interval=500)

plt.show()

ani.save('heat_udnh_fwd_error.gif', fps=24)
print("Animation saved successfully as heat_udnh_fwd_error (mp4 and gif)")

### Crank-Nicolson

#### Computations

In [ ]:
u = np.zeros((N+1,M+1))
u[:,0] = f(x)
ug0 = g0(t)
ug1 = g1(t)

In [ ]:
cn_matrix_left = np.diag(2*(1+s)*np.ones(N+1))

for j in range(1,N):
    cn_matrix_left[j,j+1]=-s
    cn_matrix_left[j,j-1]=-s

cn_matrix_left[0,0]=1
cn_matrix_left[-1,-1]=1

In [ ]:
cn_matrix_left

In [ ]:
cn_matrix_right = np.diag(2*(1-s)*np.ones(N+1))

for j in range(1,N):
    cn_matrix_right[j,j+1]=s
    cn_matrix_right[j,j-1]=s

cn_matrix_right[0,0]=0
cn_matrix_right[-1,-1]=0

In [ ]:
cn_matrix_right

In [ ]:
for k in range(M):
    utemp = u[:,k]
    utemp = cn_matrix_right@utemp
    g = np.zeros(N+1)
    g[0]=ug0[k+1]
    g[-1]=ug1[k+1]
    utemp = solve(cn_matrix_left, utemp+g)
    u[:,k+1] = utemp

In [ ]:
u_cn = u[:,:]

In [ ]:
error_cn = u_cn[:,:]-udnh[:,:]

In [ ]:
print(f'Linf error = {norm(error_cn[:,M],ord=np.inf)}')

In [ ]:
df_cn = pd.DataFrame(u_cn.T, columns = x, index = t)

In [ ]:
df_cn

In [ ]:
df_cn_error = pd.DataFrame(error_cn.T, columns = x, index = t)

In [ ]:
df_cn_error

#### Graphs

In [ ]:
fig,ax = plt.subplots(subplot_kw={"projection": "3d"},figsize=(10,10), layout='tight')
ax.plot_surface(X, T, u_cn.T,cmap=cm.viridis)
ax.set(title='Crank-Nicolson approximation',xlabel='x',ylabel='t',zlabel='u')
ax.set_box_aspect(None, zoom=0.87)
plt.show()

In [ ]:
fig,ax = plt.subplots()
ct = ax.contourf(X, T, u_cn.T, levels = levels, extend = 'both');
fig.colorbar(ct)
ax.set(title='Temperature contour map',xlabel='x',ylabel='t')
plt.show()

In [ ]:
fig,ax = plt.subplots()
for k in range(0,M+1,M//5):
    ax.plot(x,u_cn[:,k],label=f't = {t[k]}')
ax.set(title='Temperature variation over time',xlabel='x',ylabel='u')
ax.grid()
ax.legend()
plt.show()

In [ ]:
fig,ax = plt.subplots()
ax.plot(x,u_cn[:,M],label=f'Approximation at t = {t[M]}')
ax.plot(x,udnh[:,M],label=f'Exact solution at t = {tmax}')
ax.set(title='Comparison between exact and approximate solution',xlabel='x',ylabel='u')
ax.grid()
ax.legend()
plt.show()

In [ ]:
fig,ax = plt.subplots()
ax.plot(x,abs(error_cn[:,M]))
ax.set(title='Absolute error',xlabel='x',ylabel='error')
ax.grid()
plt.show()

In [ ]:
fig, ax = plt.subplots()

heat_approx, = ax.plot(x,u_cn[:,0],label='Crank-Nicolson approximation');
heat_exact, = ax.plot(x,udnh[:,0],linestyle='-.',label='Exact solution');

ax.legend()
ax.set(xlabel='x',ylabel='u(x,t)',xlim=xlim,ylim=ylim)
ax.grid()

def animate(k):
    heat_approx.set_ydata(u_cn[:,k])
    heat_exact.set_ydata(udnh[:,k])
    ax.set(title=f"Exact solution vs Crank-Nicolson approximation at t = {k*dt:.3f}")
    return heat_approx,heat_exact

ani = animation.FuncAnimation(fig, animate, frames=M+1)#, interval=500)

plt.show()

ani.save('heat_udnh_cn.gif', fps=24)
print("Animation saved successfully as heat_udnh_cn.gif")

In [ ]:
fig, ax = plt.subplots()

error, = ax.plot(x,abs(error_cn[:,0]));

#ax.legend()
ax.set(xlabel='x',ylabel='error',xlim=xlim,ylim=[0,3.5e-3])
ax.grid()

def animate(k):
    error.set_ydata(abs(error_cn[:,k]))
    ax.set(title=f"Error propagation. t = {k*dt:.3f}")
    return error

ani = animation.FuncAnimation(fig, animate, frames=M+1)#, interval=500)

plt.show()

ani.save('heat_udnh_cn_error.gif', fps=24)
print("Animation saved successfully as heat_udnh_cn_error.gif")

## Method comparison

In [ ]:
fig,ax = plt.subplots()
ax.plot(x,u_euler_fwd[:,M],label=f'Forward Euler method at t = {t[M]}')
ax.plot(x,u_cn[:,M],label=f'Crank-Nicolson method at t = {t[M]}')
ax.plot(x,udnh[:,M],label=f'Exact solution at t = {tmax}')
ax.set(title='Comparison between exact and approximate solution',xlabel='x',ylabel='u')
ax.grid()
ax.legend()
plt.show()